# <font face="Verdana" size=6 color='#6495ED'>  THIS IS ME

<font face="Verdana" size=3 color='#40E0D0'> Profs. Larissa Driemeier, Thiago Martins

<center><img src='https://drive.google.com/uc?export=view&id=1LYiTAE2KG5dJf_qIoVKOluUhzrK-AmfP' width="600"></center>

Na próximas duas aulas, vocês irão construir um programa para reconhecimento facial de qualquer colega seu do curso de IA. Para isso, serão 3 etapas:
1. Geração de Eigenfaces - utilização do banco de dados *Labeled Faces in the Wild* (LFW) em conjunto com as fotos de seus colegas, para extrair as características principais de uma face humana via PCA;
2. Criação um classificador SVM para reconhecimento facial de seus colegas;
3. Utilização do classificador em tempo real na competição em sala.

Prontos? Esta é a __primeira__ etapa.

In [ ]:
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
import matplotlib.pyplot as plt
import html
import pandas as pd
import seaborn as sns
from google.colab import files
import os
from pathlib import Path
import subprocess
from typing import Tuple, List, Optional

# <font face="Verdana" size=6 color='#6495ED'>   LFW

*Labeled Faces in the Wild* (LFW) é um banco de dados de fotografias de rostos projetado para estudar o problema do reconhecimento facial. Atualmente, quatro conjuntos diferentes de imagens LFW,  incluindo o original e três tipos diferentes de imagens "alinhadas". De acordo com os pesquisadores, as imagens com alinhamento tipo afunilamento profundo (*deep funneling*) produziram resultados superiores para a maioria dos algoritmos de verificação de rosto em comparação com os outros tipos de imagem. Portanto, o conjunto de dados carregado aqui é essa versão.

<center><img src='https://drive.google.com/uc?export=view&id=1IGeppDQn5xMl8k1uidUjTqahKPT7n0dt' width="400"></center>

<small> Imagem extraída do [link](https://github.com/dribnet/lfw_fuel).

Abaixo, o conjunto de dados de pessoas LFW é carregado. Os principais parâmetros são `min_faces_per_person` e `resize`. O primeiro parâmetro indica o mínimo de imagens que uma pessoa deve ter para ser selecionada para o dataset e o segundo parâmetro indica a proporção na qual a imagem é redimensionada.
```
lfw_people = fetch_lfw_people(resize=1.0)
```

As imagens são $125 \times 94 = 11750$. Você pode, por exemplo,
escolher um `resize` de 0.5, que leva a uma dimensão total de $62\times 47 = 2914$ para cada ponto do espaço (imagem). Além disso, a escolha default  `min_faces_per_person = None` levou a um conjunto de 13233 imagens no total.

In [ ]:
from sklearn.datasets import fetch_lfw_people
lfw_people = fetch_lfw_people(resize=1.0)

Perceba com o código abaixo que a variável `lfw_people.images` contém os dados referentes às imagens e a variável `lfw_people.target` contém um número referente à identificação da imagem (0-5748). A conexão entre os números usados na identificação e o nome de cada pessoa é obtida através da variável `lfw_people.target_names`.

A matriz 2D `lfw_people.data` contém o número de linhas equivalente ao número de imagens e o número de colunas equivalente à dimensão (número de características) de cada imagem.

In [ ]:
cont_imagens, altura_lfw, largura_lfw = lfw_people.images.shape
image_shape = (altura_lfw, largura_lfw)
n_classes = lfw_people.target_names.shape[0]

print(f'O dataset contém ',cont_imagens, 'imagens de dimensões ', altura_lfw, 'por', largura_lfw)
print(f'Portanto, o número de características (dimensão) de cada imagem é {altura_lfw} x {largura_lfw} = {lfw_people.data.shape[1]}')
print(f'Portanto, todos os dados estão resumidos em uma matriz de {lfw_people.data.shape[0]} linhas por {lfw_people.data.shape[1]} colunas.\n')

print(f'Cada uma das imagens tem um número de identificação. Os números estão armazenados em um vetor de dimensão {lfw_people.target.shape[0]}:')
print(f'{lfw_people.target}\n' )

print(f'O número direciona a imagem a uma das {n_classes} pessoas diferentes existentes no dataset. São elas:')
target_names = lfw_people.target_names
print(f'{target_names}\n')

plt.imshow(lfw_people.images[-1], cmap='gray')
name_ex = lfw_people.target_names[lfw_people.target[-1]]

plt.title('Exemplo '+str(lfw_people.target[-1])+': '+str(name_ex))
plt.show()




Junichiro Koizumi foi um político japonês que serviu como Primeiro-Ministro do Japão de 2001 a 2006. Ele é conhecido por suas políticas reformistas e estilo de liderança carismático. Koizumi nasceu em 8 de janeiro de 1942, em Yokosuka, na província de Kanagawa. Ele ingressou na política seguindo os passos de seu pai e avô, ambos políticos influentes.

In [ ]:
for (i,j) in zip(range(len(target_names)),target_names):
  if j == "Roberto Carlos":
    print(i,j)
  if j == "Tony Blair":
    print(i,j)
  if j == "Halle Berry":
    print(i,j)
  if j == "Pele":
    print(i,j)
  if j == "Naomi Campbell":
    print(i,j)
  if j == "Colin Powell":
    print(i,j)
  if j == "Margaret Thatcher":
    print(i,j)
  if j == "Gisele Bundchen":
    print(i,j)
  if j == "Tom Cruise":
    print(i,j)

Veja que o dataset é composto das mais diversas faces...

<center><img src='https://drive.google.com/uc?export=view&id=1MqvGqzlLo98WjNkhO9dsAqTGH4HN8x8o' width="900"></center>

In [ ]:
plt.figure(figsize=(20,120))
face_numbers = [1047,1918,3486,4010,5406]
for i in range(len(face_numbers)):
    plt.subplot(1,5,i+1)
    for j in range(len(lfw_people.target)):
      if lfw_people.target[j] == face_numbers[i]:
        plt.imshow(lfw_people.images[j], cmap='gray')
        plt.title('imagem '+str(j)+':\n '+str(target_names[lfw_people.target[j]]))
        plt.xticks(())
        plt.yticks(())
plt.show()

 Veja outros exemplos...

In [ ]:
plt.figure(figsize=(25,25))

for i in range(25):
    plt.subplot(5,5,i+1)
    plt.imshow(lfw_people.images[i*40], cmap='gray')
    plt.title('imagem '+str(i*40)+': '+str(target_names[lfw_people.target[i*40]]))
    plt.xticks(())
    plt.yticks(())
plt.show()

# <font face="Verdana" size=6 color='#6495ED'>   PCA
Lembram-se das aulas de decomposição em valores singulares?


Seja $X_{n \times m}$ um conjunto de $n$ dados com dimensão $m$.


O vetor médio do conjunto $X$ é um vetor de $m$ componentes que são dadas por:
\begin{equation}
\bar{X}_i = \frac{1}{n}\sum_{k=1}^n X_{k, i}
\end{equation}

A *matriz de covariância* do conjunto de dados $X$ é dada por:

\begin{equation}
\mbox{cov}\left[X, X\right]_{i,j} = \frac{1}{n}\sum_{k=1}^n (X_{k,i}-\bar{X_i})(X_{k,j}-\bar{X_j})
\end{equation}

Em particular, o *traço* desta matriz, ou seja, a soma dos componentes de sua diagonal principal, é o valor médio da distância ao quadrado dos pontos em $X$ ao ponto médio.

Nota-se que esta é uma matriz simétrica e positiva semi-definida.

A dimensão $m$ é frequentemente *redundante*, ou seja, seria possível descrever um ponto do conjunto por uma quantidade menor de parâmetros.

Suponha uma transformação linear $W_{l \times m}$ com $l<m$ que transforma um ponto no espaço de dimensão $m$ para um ponto no espaço de dimensão $l$. Os pontos se transformam com $Y_i = W (X_i - \bar{X})$.

É razoável supor que a transformação que captura o *máximo* de informação sobre $X_{n \times m}$ é a que produz o conjunto de dados $Y_{n \times l}$ com *máxima* covariância em suas colunas.

Tal transformação pode ser obtida pelos primeiros $l$ componentes (ordenados em ordem decrescente de autovalores) da *decomposição espectral* da matriz $\mbox{cov}\left[X, X\right]$.

De fato, pelo teorema espectral, toda matriz real simétrica positiva $m\times m$ tem $m$ autovalores positivos e todos os seus autovetores são ortogonais.

A matriz $W$ é assim formada pelos primeiros $l$ componentes de tal decomposição.

Deste modo, é possível reduzir um vetor com $m$ componentes para um vetor com $l$ componentes retendo-se o *máximo* de informação.

## O PCA em sklearn

O objeto `PCA` da biblioteca `sklearn.decomposition` realiza a **decomposição em componentes principais (Principal Component Analysis)** de um conjunto de dados.

A sintaxe básica do construtor é:
```
from sklearn.decomposition import PCA
pca = PCA(n_components=k)
```
O parâmetro opcional `n_components` define o número de componentes principais a serem mantidos.  
Se for omitido, o PCA manterá todos os componentes (igual ao número de características do conjunto de dados).

O método `fit(dados)` ajusta o modelo aos dados, onde `dados` é uma matriz NumPy de tamanho `(n_amostras, n_características)` — ou seja, cada linha representa um ponto no espaço de características.

Após o ajuste, o objeto `pca` possui os seguintes atributos importantes:
- `mean_`: vetor com a **média de cada característica** (centro dos dados);
- `components_`: matriz cujas **linhas são os autovetores** da matriz de covariância — ou seja, os componentes principais;
- `explained_variance_`: vetor com os **autovalores** da matriz de covariância, que indicam a variância explicada por cada componente;
- `explained_variance_ratio_`: vetor com a **fração da variância total explicada** por cada componente.


In [ ]:
from sklearn.decomposition import PCA

# Cria uma matriz de dados 3 (exemplos) x 2 (características)
dados = np.array([[1, 2], [3, 4], [5, 6]])


# Gráfico dos dados
plt.scatter(dados[:,0], dados[:,1], color='crimson', marker='o', label = 'Dados')

# Ajuste de reta
coef = np.polyfit(dados[:,0], dados[:,1], deg=1)
reta = np.poly1d(coef)
x_reta = np.linspace(min(dados[:, 0]), max(dados[:, 0]), 5)
y_reta = reta(x_reta)
plt.plot(x_reta, y_reta, color='navy', linestyle='--', label='Regressão linear')

plt.title('Dados')
plt.ylabel(r'$x_2$')
plt.xlabel(r'$x_1$')
plt.axis('equal')
plt.legend()
plt.grid(True)
plt.show()

# Cria uma instância da classe PCA especificando o número de componentes:
pca = PCA(n_components=2)

# Ajusta a PCA no conjunto de dados
pca.fit(dados)

# Acessa o ponto médio
media = pca.mean_
print("Ponto médio:\n", media)

# Acessa a variância explicada através dos autovetores
autovalores = pca.explained_variance_
print("Autovalores:\n", autovalores)

# Variância explicada
print("Variância explicada:\n", pca.explained_variance_ratio_)

# Acessa os componentes principais (vetores de projeção) através dos autovetores
componentes_principais = pca.components_
print("Componentes principais:\n", componentes_principais)

# Transforma os dados originais usando a projeção PCA
dados_transformados = pca.transform(dados)
print("Dados transformados:\n", dados_transformados)
print("\n")

# Transforma os dados originais usando a projeção PCA
dados_recuperados = pca.inverse_transform(dados_transformados)
print("Dados recuperados:\n", dados_recuperados)
print("\n")


Veja que no conjunto de dados acima os pontos são perfeitamente alinhados em uma linha reta em um espaço 2D. O PCA deste conjunto de dados, portanto, produziu dois autovalores:
* Um autovalor positivo, correspondente à direção ao longo da linha (onde há variação).
* Um autovalor que é zero, correspondente à direção perpendicular à linha (onde não há variação).

<center><img src='https://drive.google.com/uc?export=view&id=19IUASr4WgNh_0mDDbDPO9-NdfMaRiN3c' width="600"></center>

Obviamente, recupera-se totalmente a matriz de dados porque foram utilizados todas as componentes principais. Troque o número de componentes e verifique que o resultado será o mesmo por causa do alinhamento dos dados.

Modifique um pouco a matriz `dados`, como por exemplo:
```
dados = np.array([[1, 2], [3, 4.8], [5, 6]])
```

para desalinhar os pontos e verifique o resultado.


# <font face="Verdana" size=6 color='#6495ED'> Eigenfaces

Eigenfaces é um processo para representar faces humanas, tradicionalmente para reconhecimento.
O princípio por trás do Eigenfaces é gerar uma representação de imagens de rostos baseada em componentes principais de uma amostra grande o suficiente de rostos.

A hipótese subjacente é que a amostra tem características suficientes para representar de forma fidedigna mesmo rostos que não pertencem a ela.

Como visto, as imagens são representadas por matrizes.
Estas matrizes serão "linearizadas" em vetores para serem usadas como variáveis explicativas (usando ```reshape```).

A totalidade de imagens será colocada na variável ```ìmages```.

Finalmente, iremos dividir esta variável em um conjunto de "treinamento" em ```X_train``` e um de "testes" em ```X_test``` em uma divisão de 90%/10%.
Os rótulos também serão divididos, mas não serão relevantes para nossa atividade aqui.

In [ ]:
from sklearn.model_selection import train_test_split
np.random.seed(10)
# Constroi vetores de imagens e rótulos (estes último)
images = []
labels = []
for i, t in zip(lfw_people.images, lfw_people.target):
  images.append(i.reshape(-1)) # lineariza matrizes
  labels.append(t)
images = np.array(images)
labels = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.1)

print(f"Dimensões de images:{images.shape}")
print(f"Dimensões de X_train:{X_train.shape}")
print(f"Dimensões de X_test:{X_test.shape}")

Faremos a decomposição espectral do conjunto de "treinamento" com 1000 componentes (!!!). Note que esta quantidade de componentes é _excessiva_ para propósitos de identificação, e que o cálculo dessa decomposição pode O O resultado da chamada a  ```fit``` será atribuido à variável ```eigenfaces```.

In [ ]:
eigenfaces = PCA(n_components=1000).fit(X_train)

Mesmo considerando que o conjunto original tem mais de 10 mil variáveis, essas mil componentes devem capturar mais de 99% da variância da amostra.
Verifique isso executando esta célula:

In [ ]:
print(f"Dimensão original: {X_train.shape[-1]}")
print(f"Variância explicada com 1000 componentes: {float(eigenfaces.explained_variance_ratio_.sum())}")

A imagem média está armazenada no atributo ```mean_```. Podemos mostrá-la com o comando:

In [ ]:
plt.imshow(eigenfaces.mean_.reshape(image_shape), cmap='gray')

Podemos verificar a capacidade da decomposição em representar imagens de fora da base de treinamento. Vamos pegar a primeira (de forma arbitrária, qualquer uma deveria funcionar) imagem do conjunto de testes e colocá-la na variável image_test. Podemos mostrar essa imagem:

In [ ]:
image_test = X_test[0]
plt.imshow(image_test.reshape(image_shape), cmap='gray')

O comando transform projeta a imagem no espaço de eigenfaces. Mostre que o resultado é um vetor de mil componentes. Em seguida, o comando inverse_transform reconstroi a imagem.
Observe o resultado. Lembre-se que essa imagem está *fora* do conjunto de treinamento de eigenfaces


In [ ]:
im_transform = eigenfaces.transform([image_test])
print(im_transform.shape)
im_reconstruct = eigenfaces.inverse_transform(im_transform)
plt.imshow(im_reconstruct.reshape(image_shape), cmap='gray')

# <font face="Verdana" size=6 color='#6495ED'>  CRIAÇÃO DO DATASET

Veja que os dados LFW estão resumidos em uma matriz de $13233$ linhas (número total de imagens) por $11750$ colunas (dimensão das imagens, se você usou `resize=1.0`).  As imagens têm dimensão de $125 \times 94$.

Ao calcular o PCA, seja apenas com o conjunto original LFW ou incluindo nossas imagens, é essencial que todas tenham a mesma dimensão.

Então, a parte de criação do dataset é dividida em duas partes:
1. organização das imagens
2. Criação do dataset




## Organização das imagens

O código abaixo é um tutorial para correção das fotos para a dimensão correta de $125 \times 94$ centralizando o rosto. Ao final, o código ensina a salvar a foto corrigida em seu computador local.

<center><img src='https://drive.google.com/uc?export=view&id=14s6TEFNqlr5kglKnBtIrMj-nNQ8w3IWq' width="2000"></center>

Ao final, após redimensionar e salvar todas as suas imagens localmente, você deve atualizá-las no subdiretório correspondente do seu Google Drive, conforme a URL que você forneceu na planilha disponibilizada neste [link](https://docs.google.com/spreadsheets/d/1mnkR0IN2fKe_-oifhi87V2WmF9wLsJaT36_GT-MTTSM/edit?usp=sharing).



In [ ]:
!gdown https://drive.google.com/file/d/1zEr_OV7UA4p3wDqDmqiDxyLotW-w-sFb/view?usp=sharing --fuzzy

In [ ]:
name = 'Larissa_Driemeier_01.jpg'
image_org = cv2.cvtColor(cv2.imread(name), cv2.COLOR_BGR2RGB)
plt.imshow(image_org),plt.title('Larissa'),plt.axis('off')
# Imagem em escala de cinzas
image = cv2.cvtColor(image_org,cv2.COLOR_RGB2GRAY)
image_org = cv2.cvtColor(image_org,cv2.COLOR_RGB2GRAY)
x,y = image.shape[0], image.shape[1]

### Haar Cascade
A linha de código abaixo utiliza o Haar Cascade, algoritmo pioneiro na detecção de rostos em tempo real (muito antes do Deep Learning se tornar famoso), e ainda é usado por sua simplicidade e velocidade, principalmente com OpenCV. A técnica foi introduzida por **Viola e Jones (2001)** no artigo:
> *"Rapid Object Detection using a Boosted Cascade of Simple Features"*

O algoritmo usa essas Haar-like features junto com um classificador em cascata e boosting (AdaBoost). Portanto, o nome **Haar Cascade** vem da combinação de duas ideias principais:

#### Haar Features

São padrões simples de contraste entre regiões claras e escuras da imagem, inspirados nas *funções de Haar (wavelets)*.   No rosto, temos, pore xemplo, regiões escuras nos olhos comparadas com bochechas claras, sombra do nariz etc.

Essas features são calculadas rapidamente usando *diferenças de soma de pixels em áreas retangulares*.

#### Cascade de Classificadores
Em vez de um único classificador complexo, o algoritmo usa uma *sequência (cascata)* de classificadores simples. Cada etapa descarta regiões improváveis de conter um rosto. Apenas as mais promissoras passam adiante, acelerando o processo.

In [ ]:
from google.colab.patches import cv2_imshow
haar_face_cascade = cv2.CascadeClassifier(cv2.samples.findFile(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'))

In [ ]:
# Reconhecimento de faces
rostos = haar_face_cascade.detectMultiScale(image)
print(rostos)
print(f'{rostos.shape[0]} rosto(s) detectado(s)')
n_rostos = rostos.shape[0]

print(f'\n A matriz rostos:\n {rostos}')

Veja que o rosto detectado se inicia na posição `rostos[0],rostos[1]` e tem dimensão `rostos[2] x rostos[3]`.

O código abaixo posiciona um retângulo ao redor das faces reconhecidas na imagem.

In [ ]:
# Colocando o retângulo ao redor da face reconhecida
for (x, y, largura, altura) in rostos:
  image_r = cv2.rectangle(image,(x,y),(x+largura,y+altura),(255,0,0),2)
#Plotando a imagem
plt.imshow(image_r, cmap='gray'),plt.title('Rosto(s) Detectado(s)');

Caso tenha mais de uma face, o código abaixo mostra, em zoom, todos os rostos detectados.

In [ ]:
# Zoom no rosto detectado

plt.figure(figsize=(10,10)) # especificação do tamanho total do grid
i=0
for (x, y, largura, altura) in rostos:
  i += 1
  plt.subplot(1,n_rostos,i)    # o número máximo de imagens é 5x5=25
  plt.imshow(cv2.cvtColor(image[y:y+altura, x:x+largura], cv2.COLOR_BGR2RGB))
  plt.title('imagem '+ str(i)+'   largura: '+str(largura)+'x altura:'+str(altura))
plt.show()

Se o código detectou mais de um rosto, escolha o maior. Esta imagem que deverá ser reconhecida!!!'



In [ ]:
#se houver vários rostos, escolha o quadro desenhado ao redor da maior imagem,
x, y, largura, altura = rostos[rostos[:,-1].argsort()[-1]] #
print(x,y,largura,altura)
plt.imshow(image[y:y+altura, x:x+largura], cmap='gray'),plt.title('Rosto Escolhido');
print(f'A imagem mostrada tem {largura} pixels de largura e {altura} pixels de altura.')

Se essa é a imagem que deve ser reconhecida, então precisamos transformá-la para nosso tamanho padrão, ié, `altura_lfw = 125` e  `largura_lfw = 94`.

Mais do que isso, para casar perfeitamente com o alinhamento *funneled* do LFW, é ideal reconhecer os olhos e através de uma transformada afim, alinhá-los de forma a repetir a posição desta base.

O código abaixo procura reenquadrar fotos, re-alinhálas através da detecção de olhos e equalizar os tons de cinza e iluminação.

Os níveis de alinhamento são:


0. Nenhuma face detectada, alinhamento feito simplesmente pelo centro da imagem (provavelmente inútil para treinamento).
1. Rosto detectado, mas não olhos. Alinhado pelo centro da face, mas sem rotações (não é ideal para treinamento).
2. Rosto e olhos detectados, mas sem posicionamento fino. Alinhamento pelos olhos (bom para treinamento).
3. Rosto e olhos detectados com posicionamento fino. Alinhamento pelos olhos (excelente para treinamento).


In [ ]:
LFW_L_EYE = (105, 110)
LFW_R_EYE = (145, 110)
OUTPUT_SHAPE = (94, 125) # (width, height)
ASPECT_RATIO = OUTPUT_SHAPE[0] / OUTPUT_SHAPE[1]

# Initialize classifiers once
face_cascade = cv2.CascadeClassifier(cv2.samples.findFile(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'))
eye_cascade = cv2.CascadeClassifier(cv2.samples.findFile(cv2.data.haarcascades + 'haarcascade_eye_tree_eyeglasses.xml'))
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def get_hough_pupil(eye_roi: np.ndarray) -> Optional[Tuple[int, int]]:
    """Attempts to find the precise pupil center using Hough Circles."""
    h, w = eye_roi.shape

    # Tighten ROI to avoid eyebrows/lower lids based on your specs
    crop_y1, crop_y2 = int(h * 0.15), int(h * 0.85)
    crop_x1, crop_x2 = int(w * 0.10), int(w * 0.90)
    tight_roi = eye_roi[crop_y1:crop_y2, crop_x1:crop_x2]

    min_r = int(w * 0.10)
    max_r = int(w * 0.20)
    circles = cv2.HoughCircles(
        tight_roi,
        cv2.HOUGH_GRADIENT,
        dp=1.2,                # Fixed: Don't oversample the accumulator
        minDist=int(w * 0.5),  # Keep high to force single circle
        param1=60,             # Slightly more forgiving Canny
        param2=15,             # avoid false positives
        minRadius=min_r,
        maxRadius=max_r
    )

    if circles is not None:
        # Get the first circle, convert coords back to the original eye_roi space
        cx, cy, _ = np.round(circles[0][0]).astype("int")
        rgb_img = cv2.cvtColor(tight_roi, cv2.COLOR_GRAY2RGB)
        cv2.circle(rgb_img,(int(circles[0,0,0]),int(circles[0,0,1])),int(circles[0,0,2]+0.5),(0,255,255),1)
        return (cx + crop_x1, cy + crop_y1)
    return None

def align_and_normalize(image: np.ndarray, left_eye: Tuple[float, float], right_eye: Tuple[float, float]) -> np.ndarray:
    """Performs the affine warp, LFW crop, and CLAHE normalization."""
    src_pts = np.array([left_eye, right_eye], dtype=np.float32)
    dst_pts = np.array([LFW_L_EYE, LFW_R_EYE], dtype=np.float32)

    transform_matrix, _ = cv2.estimateAffinePartial2D(src_pts, dst_pts)
    warped = cv2.warpAffine(image, transform_matrix, (250, 250))

    if len(warped.shape) == 3:
        warped = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)

    cropped = warped[70:195, 78:172]
    resized = cv2.resize(cropped, OUTPUT_SHAPE, interpolation=cv2.INTER_AREA)
    return clahe.apply(resized)

def center_crop_and_normalize(image: np.ndarray, center: Tuple[int, int], height: int) -> np.ndarray:
    """Fallback cropper for L1 and L0."""
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    cx, cy = center
    width = int(height * ASPECT_RATIO)

    # Calculate bounding box, bounded by image dimensions
    x1 = max(0, cx - width // 2)
    y1 = max(0, cy - height // 2)
    x2 = min(image.shape[1], cx + width // 2)
    y2 = min(image.shape[0], cy + height // 2)

    cropped = image[y1:y2, x1:x2]
    resized = cv2.resize(cropped, OUTPUT_SHAPE, interpolation=cv2.INTER_AREA)
    return clahe.apply(resized)

def process_face(image: np.ndarray) -> Tuple[int, np.ndarray]:
    """
    Attempts hierarchical face processing.
    Returns: (level_achieved, normalized_image_tensor)
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image.copy()

    # --- LEVEL 1-3 ENTRY: Face Detection ---
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))

    if len(faces) == 0:
        # --- LEVEL 0 FALLBACK: Blind Center Crop ---
        h, w = gray.shape
        # Use a reasonable height assumption (e.g., half the image height)
        return 0, center_crop_and_normalize(gray, (w//2, h//2), h // 2)

    # Assume the largest detection is the target face
    fx, fy, fw, fh = max(faces, key=lambda rect: rect[2] * rect[3])
    face_roi = gray[fy:fy+fh, fx:fx+fw]

    # --- LEVEL 2-3 ENTRY: Eye Detection ---
    eyes = eye_cascade.detectMultiScale(face_roi, scaleFactor=1.1, minNeighbors=5)

    # We need exactly 2 eyes for a valid alignment
    if len(eyes) >= 2:
        # Sort eyes by X coordinate to reliably separate Left (viewer's left) and Right
        eyes = sorted(eyes, key=lambda e: e[0])
        e1, e2 = eyes[0], eyes[-1] # Take leftmost and rightmost if > 2 detected

        # Calculate bounding box centers
        l_eye_haar = (fx + e1[0] + e1[2]//2, fy + e1[1] + e1[3]//2)
        r_eye_haar = (fx + e2[0] + e2[2]//2, fy + e2[1] + e2[3]//2)

        # --- LEVEL 3 ENTRY: Hough Transform ---
        l_roi = face_roi[e1[1]:e1[1]+e1[3], e1[0]:e1[0]+e1[2]]
        r_roi = face_roi[e2[1]:e2[1]+e2[3], e2[0]:e2[0]+e2[2]]

        l_pupil = get_hough_pupil(l_roi)
        r_pupil = get_hough_pupil(r_roi)

        if l_pupil and r_pupil:
            # Map pupil coords back to global image space
            l_eye_global = (fx + e1[0] + l_pupil[0], fy + e1[1] + l_pupil[1])
            r_eye_global = (fx + e2[0] + r_pupil[0], fy + e2[1] + r_pupil[1])
            return 3, align_and_normalize(image, l_eye_global, r_eye_global)

        # --- LEVEL 2 FALLBACK: Haar Eye Centers ---
        return 2, align_and_normalize(image, l_eye_haar, r_eye_haar)

    # --- LEVEL 1 FALLBACK: Face Bounding Box Only ---
    # Center of the face bounding box
    face_center = (fx + fw // 2, fy + fh // 2)
    # Use the bounding box height, slightly expanded, to mimic LFW scaling
    return 1, center_crop_and_normalize(gray, face_center, int(fh * 1.2))

In [ ]:
lvl, imgproc = process_face(image_org)
print(f"Nível de alinhamento obtido: {lvl}")
plt.imshow(imgproc, cmap='gray'),plt.title('Rosto Alinhado');

Podemos ver como fica esta imagem no espaço de eigenfaces.
Precisamos transformar esta imagem em um vetor linear. Numpy faz isso com o invocando-se o método reshape(1,-1)[0]

In [ ]:
vetor_imgproc = imgproc.reshape(1,-1)[0]
im_transform = eigenfaces.transform([vetor_imgproc])
print(im_transform.shape)
im_reconstruct = eigenfaces.inverse_transform(im_transform)
plt.imshow(im_reconstruct.reshape(image_shape), cmap='gray')

## Dataset

O tutorial abaixo permite que vocês o dataset com as fotos dos colegas. Para isto, as etapas são:
* download download da planila `ThisIsMe.csv` do google drive, onde todos seus colegas e você compartilharam as fotos - [link](https://docs.google.com/spreadsheets/d/1MmfWuFmUT42y--VwyFFcf79X7lhHeT6U5XNSR7ZC_o4/edit?usp=sharing).
* upload da planilha aqui
* geração dos seguintes conjuntos de dados:
  * `turma_data` (dataset com entradas) matriz de dimensão $total \text{ }de\text{ } fotos \times 11750$, isto é, cada linha é uma foto e cada coluna um pixel da foto.
  * `turma_target` (dataset com saídas) vetor com dimensão $total\text{ } de \text{ }fotos \times 1$. Cada linha $i$ corresponde ao ID da pessoa que aparece na foto da linha $i$ do conjunto `turma_data`
  *  `turma_target_names` é um vetor com dimensão $total\text{ } de\text{ } colegas \times 1$. Cada linha $j$ corresponde ao nome da pessoa com o ID $j$.

  <center><img src='https://drive.google.com/uc?export=view&id=1J2jwPqWP3G7ipsFxldfGVbcl_R4OQLAo' width="900"></center>


Faça download da planila `ThisIsMe.csv` do google drive, onde todos seus colegas e você compartilharam as fotos, e faça upload aqui.

In [ ]:
df_ThisIsMe = pd.read_csv('https://docs.google.com/spreadsheets/d/1R03NsIjDOdcxA3RhHP4Ik3AhlFcVZImTD2dgUtL5j9s/export?format=csv')
print(df_ThisIsMe)

In [ ]:
df_ThisIsMe

O script abaixo percorre o dataFrame `df_ThisIsMe` com nomes e links de pastas do Google Drive, e para cada nome (chamado de `competidor`):

1. Cria uma subpasta com seu nome dentro da pasta ThisIsMe/;
2. Baixa o conteúdo da pasta do Drive (via gdown);
3. Salva o caminho da pasta local em uma lista diretorios.

In [ ]:
diretorios = []

for idx, competidor in df_ThisIsMe.iterrows():   # linha por linha
  nome = competidor["Nome"]
  url = competidor["URL"]
  # Cria subdiretorio

  out = "ThisIsMe/" + nome
  print(out)
  print(url)
  Path(out).mkdir(parents=True, exist_ok=True)
  # Invoca gdown via linha de comando, o programa separado é mais robusto
  subprocess.run(["gdown", url, "--fuzzy", "-O", out+"/archive.zip"], check=True)
  diretorios.append(out)

Os arquivos ainda estão compactados, o código a seguir os descompacta

In [ ]:
for d in diretorios:
  subprocess.run(["unzip", "-j", "archive.zip"], cwd=d)
  subprocess.run(["rm", "archive.zip"], cwd=d)

Veja os diretórios que temos:

In [ ]:
diretorios

Agora criamos uma lista com os nomes de cada um, que chamaremos de `turma_target_names`.

In [ ]:
turma_target_names = os.listdir('ThisIsMe/')
print(turma_target_names)

O número de clases que teremos corresponde ao tamanho da lista que acabams de criar.

In [ ]:
#turma_target_names.remove('.ipynb_checkpoints') # remova o comentário, se necessário.
n_classes = len(turma_target_names)
print(f'Nosso problema terá {n_classes} classes.')

Vamos trabalhar no diretório base chamado `ThisIsMe`, que criamos. Você deve ter um diretório com seu nome.

Veja que o código abaixo percorre os diretórios contendo imagens de cada um (cujos nomes já estão na lista `turma_target_names`) e executa os seguintes passos:

1. Define as dimensões das imagens com base no dataset LFW (`altura_lfw` e `largura_lfw`).
2. Procura imagens com extensão `.jpg` ou `.jpeg` em cada subpasta correspondente a um nome.
3. Lê cada imagem em **escala de cinza** usando `cv2.imread(...)`.
4. Achata a imagem para um vetor 1D com `reshape(1, -1)[0]` e armazena em `turma_data`.
5. Registra o **rótulo da pessoa** (índice do nome na lista) no vetor `turma_target`.

Ao final, teremos duas listas:
- `turma_data`: lista com todas as imagens na forma de vetores 1D.
- `turma_target`: lista com os rótulos (índices correspondentes às classes/pessoas).

In [ ]:
turma_data = []
turma_target = []
basedir="ThisIsMe/"
for i in range(len(turma_target_names)):
  print(f"{turma_target_names[i]} ({i+1}/{len(turma_target_names)})")
  included_extensions = ['jpg','jpeg', 'JPG', 'JPEG']
  imagens = [fn for fn in os.listdir(basedir+turma_target_names[i]+'/')
              if any(fn.endswith(ext) for ext in included_extensions)]
  for j in range(len(imagens)):
    print(f"{imagens[j]} ({j+1}/{len(imagens)})")
    img = cv2.imread('{0}/{1}/{2}'.format(basedir,turma_target_names[i],imagens[j]), cv2.IMREAD_GRAYSCALE)
    lvl, improc = process_face(img)
    if lvl>=1:
      turma_data.append(improc.reshape(1,-1)[0])
      turma_target.append(i)
      if lvl==1:
        print("Aviso: Rosto alinhado apenas com caixa de detecção de face!")
    else:
      print("Falha ao alinhar rosto")

Os dados de entrada são `turma_data` com número de linhas coincidente com número total de imagens por 11750 colunas (cada coluna uma característica da imagem). Os dados de saída estão armazenados no dataset `turma_target`, que tem dimensão número total de imagens, cada linha é um número inteiro que pode variar entre 0 - `tamanho da turma` (cada número representa um nome).


In [ ]:

turma_data = np.array(turma_data)
turma_target = np.array(turma_target)
print(f'Nossos dados de entrada estão em uma matriz {turma_data.shape} nomeada de turma_data.')
print(f'Nossos dados de saída estão em um vetor {turma_target.shape} nomeado de turma_target.')


In [ ]:
largura_lfw, altura_lfw = OUTPUT_SHAPE

plt.figure(figsize=(5,5))
plt.imshow(turma_data[-1].reshape(altura_lfw, largura_lfw), cmap='gray')
plt.title('imagem original')
plt.show()

In [ ]:
print(turma_target)
print(turma_target_names)

Para garantir que seus dados sejam **mantidos mesmo após encerrar o Notebook**, é importante montar o seu Google Drive no Colab. Assim, os arquivos salvos no Drive ficarão disponíveis para serem reutilizados posteriormente.

O módulo `pickle` é feito para salvar e recuperar (*serializar* e *desserializar*) qualquer estrutura de dados do Python — listas, dicionários, classes, modelos, funções, etc. Ele entende a estrutura interna desses objetos e os transforma em um formato binário que pode ser armazenado e depois reconstruído exatamente como estavam.

Isso permite salvar em Python variáveis, dados pré-processados ou modelos treinados de forma prática e reutilizável.

In [ ]:
import pickle as pk
import os

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
# Construa o caminho completo do arquivo
file_path = '/content/gdrive/MyDrive/AprendizadoMaquinasI/2026/ThisIsMe/Dataset/'

# Verifique se o diretório existe, se não, crie-o
dir_path = os.path.dirname(file_path)
if not os.path.exists(dir_path):
    os.makedirs(dir_path)

# Salve os arquivos
with open(file_path+'turma_data.pkl', 'wb') as pickle_file:
    pk.dump(turma_data, pickle_file)

with open(file_path+'turma_target.pkl', 'wb') as pickle_file:
    pk.dump(turma_target, pickle_file)

with open(file_path+'turma_target_names.pkl', 'wb') as pickle_file:
    pk.dump(turma_target_names, pickle_file)

Em outro notebook, você pode recuperar estes dados com os seguintes comandos:

In [ ]:
file_path = '/content/gdrive/MyDrive/AprendizadoMaquinasI/2026/ThisIsMe/Dataset/'

# Verifique se o diretório existe, se não, crie-o
dir_path = os.path.dirname(file_path)
if not os.path.exists(dir_path):
    print("Diretório inexistente, verifique se o google drive está montado")

# Salve os arquivos
with open(file_path+'turma_data.pkl', 'rb') as pickle_file:
    turma_data = pk.load(pickle_file)

with open(file_path+'turma_target.pkl', 'rb') as pickle_file:
    turma_target = pk.load(pickle_file)

with open(file_path+'turma_target_names.pkl', 'rb') as pickle_file:
    turma_target_names = pk.load(pickle_file)

# <font face="Verdana" size=6 color='#6495ED'> AGORA É COM VOCÊ ...

Seria interessante aplicar PCA para reduzir a dimensão dos pontos de `lfw_people.data` a uma dimensão de `n_PC` componentes retendo-se o *máximo* de informação possível. Verifique, através da plotagem da porcentagem de variação explicada acumulada, qual o valor razoável para `n_PC`. Para encontrar as direções principais, passe o parâmetro `n_components=n_PC` no construtor de `PCA` e crie uma nova decomposição de `lfw_people.data` com o método `fit`.

Depois, os dados projetados no novo espaço de dimensão reduzida são obtidos a partir de pontos no espaço original com o método `transform` (naturalmente aqui há alguma perda de informação).

Para você e seu grupo:

1. Criem o dataset:
 * `data_images` com as imagens da turma adicionadas ao `lfw_people.images`
 * `data` com os dados adicionados à lista `lfw_people.data`
 * `data_target`, acrescente mais números em `lfw_people.target`
 * `data_names`, acrescente os nomes em `lfw_people.target_names`


2. Calculem o PCA;
3. Plotem o gráfico da variância explicada acumulada e, baseando-se no gráfico, escolham um número de componentes principais a ser adotado;
4. Plotem o *rosto médio*. Uma parte interessante do PCA é que ele calcula o *rosto médio*, que pode ser interessante examinar. Isso pode ser calculado com `pca.mean_`. Este rosto mostrará a média para cada dimensão de todas as imagens no conjunto de dados. Portanto, ele efetivamente mostra um rosto MÉDIO refletindo todos os rostos no conjunto de dados.
5. Projetem o  __i-ésimo ponto__ `lfw_people.data[i]` (lembrem-se que os valores devem estar na forma de vetor linha e, portanto, dimensão de $1 \times 11750$ ) nas CPs e os projetem novamente no espaço original. Usem diferentes quantidades de componentes principais `n_PC` e vejam o quanto de informação é perdida para um baixo número de componentes consideradas. Verifiquem também, que a medida que o número de componentes consideradas cresce, a imagem recuperada se aproxima da imagem original. Um ponto do espaço original recuperado pode ser obtido a partir do espaço reduzido com o método `inverse_transform`.
6. Apenas para ilustração, remodelem as componentes principais e definam como `eigenfaces`, que é o nome dado a um conjunto de autovetores quando usado no problema de visão computacional de reconhecimento de rosto humano. Vejam que quando PCA é aplicado a dimensão de cada uma das `n_PC` componentes é de 11750 e, para mostrar a imagem,  devem remodelar o vetor para dimensão de `altura_lfw` ($125$) vs `largura_lfw` ($94$):

`eigenfaces = pca.components_.reshape((n_PC, altura_lfw, largura_lfw))`

7. Plotem as primeiras (20, por exemplo) eigenfaces e resumam suas conclusões.
8. É interessante notar que o espaço das eigenfaces não é útil apenas para representar rostos humanos, mas também pode ser usado para aproximar um cachorro (Fig. `doguito.png`) ou um cappuccino (Fig. `capuccino.png`). Vocês devem testar. Isso é possível porque as 1600 eigenfaces abrangem um grande subespaço do espaço de imagem de 32256 dimensões, correspondendo a características espaciais amplas, suaves e não localizadas, como bochechas, testa, bocas, etc.
9. Salvem os dados do PCA.

Nós começamos para você...

# <font face="Verdana" size=6 color='#6495ED'> DESAFIO

## Treinamento do Modelo SVM

Você vai criar um classificador por vetores de suporte (*Support Vector Machine* - SVM).

São várias etapas:
* O seu PCA está bom? Você explorou todas as possibilidades?
* Use o dataset de entradas (`turma_data`, imagens) e saídas (`turma_target`, números de 0 ao total de colegas). Além disso, você precisa da lista `turma_target_names`, para depois ligar os nomes aos números de classificação;
* Divida seus dados em treino e teste. Verifique se alguns colegas, que possuem poucas fotos, não ficaram somente em um grupo;
* Treine o modelo SVM, utilizando o `GridSearchCV` para ajuste de hiperperâmetros;
* Analise o desempenho do modelo, plotando, por exemplo, a matriz de confusão.

Ao final da próxima aula, você receberá um notebook onde poderá carregar seu pca:
```
pca = pk.load(open(file_path_pcs+'pca.pkl', 'rb'))
```
e seu modelo será treinado na hora, com o dataset de seus colegas.

A imagem de um colega será capturada pela câmera de seu computador, a face deverá ser identificada via Haar Cascade e, finalmente, você irá utilizar seu algoritmo para reconhecer quem é o colega. Você pode escolher 5 nomes.

```
def tirar_foto(quality=0.8, texto_botao="Capturar"):
  js = Javascript('''
    async function takePhoto(qual, texto) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = texto;
      div.appendChild(capture);

      // Abre a câmera
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      // Mostra a saída da câmera
      const video = document.createElement('video');
      video.style.display = 'block';
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;      
      await video.play();
      
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', qual);
    }
    ''')
  display(js)
  return eval_js('takePhoto({}, "{}")'.format(quality, texto_botao))

try:
  imagem_urlb64 = tirar_foto()
  imbytes = b64decode(imagem_urlb64.split(',')[1])
  im = cv2.imdecode(np.frombuffer(imbytes, dtype=np.uint8), flags=1)
  plt.imshow(im, cmap='gray'),plt.title('Imagem capturada')
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))
```



Você pode testar isso agora (é necessária uma webcam):

In [ ]:
def tirar_foto(quality=0.8, texto_botao="Capturar"):
  js = Javascript('''
    async function takePhoto(qual, texto) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = texto;
      div.appendChild(capture);

      // Abre a câmera
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      // Mostra a saída da câmera
      const video = document.createElement('video');
      video.style.display = 'block';
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', qual);
    }
    ''')
  display(js)
  return eval_js('takePhoto({}, "{}")'.format(quality, texto_botao))

In [ ]:
try:
  imagem_urlb64 = tirar_foto()
  imbytes = b64decode(imagem_urlb64.split(',')[1])
  im = cv2.cvtColor(cv2.imdecode(np.frombuffer(imbytes, dtype=np.uint8), flags=1), cv2.COLOR_RGB2BGR)
  plt.imshow(im, cmap='gray'),plt.title('Imagem capturada')
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

Vamos alinhar e processar esta foto

In [ ]:
lvl, foto_alinhada = process_face(im)
print(f"Nível de alinhamento obtido: {lvl}")
plt.imshow(foto_alinhada, cmap='gray'),plt.title('Rosto Alinhado');

Vamos projetá-la no nosso espaço de eigenfaces:

In [ ]:
vetor_imgproc = foto_alinhada.reshape(1,-1)[0]
im_transform = eigenfaces.transform([vetor_imgproc])
print(im_transform.shape)
im_reconstruct = eigenfaces.inverse_transform(im_transform)
plt.imshow(im_reconstruct.reshape(image_shape), cmap='gray')

## <font face="Verdana" size=3 color='#40E0D0'>  Data Augmentation
Aqui vale um comentário sobre *data augmentation*. Data augmentation (aumento de dados) é uma técnica usada para gerar novas imagens a partir das imagens existentes, aplicando pequenas transformações como rotações, translações, zoom, espelhamentos, etc. O objetivo é aumentar a diversidade do conjunto de treino sem coletar mais dados reais, o que ajuda a reduzir overfitting e melhora a generalização do modelo. Data augmentation é uma técnica essencial para melhorar o desempenho de modelos de aprendizado profundo, especialmente quando se trabalha com conjuntos de dados limitados.

In [ ]:
!gdown https://drive.google.com/file/d/1SiVX8Ibr6xjanDCbDqWufC9wkdXT6a-G/view?usp=sharing --fuzzy

In [ ]:
import tensorflow as tf

name = 'Albert_Einstein.jpg'
image_org = cv2.imread(name)  # Carrega no formato BGR (OpenCV padrão)
image_rgb = cv2.cvtColor(image_org, cv2.COLOR_BGR2RGB)
image_gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)  # Shape: (altura, largura)

# Adicionar canal para ficar (altura, largura, 1)
image_gray = np.expand_dims(image_gray, axis=-1)  # Novo shape: (altura, largura, 1)
image_gray = image_gray.astype('float32') / 255.0  # Normalizar [0, 1]

# Gerador de aumento para imagens em cinza
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.2,
    rotation_range=3.0,
#    brightness_range=[0.8, 1.01],
    fill_mode='nearest'
)

# 6. Gerar exemplos aumentados
augmented_images = []
for _ in range(8):  # Gerar 9 variações
    aug_img = datagen.random_transform(image_gray)
    augmented_images.append(aug_img)

# 7. Plotar resultados
plt.figure(figsize=(12, 10))
plt.suptitle('Data Augmentation em Escala de Cinzas')

# Imagem original em cinza
plt.subplot(3, 3, 1)
plt.imshow(np.squeeze(image_gray), cmap='gray')  # Remove dimensão extra para plot
plt.title('Original')
plt.axis('off')
for idx, aug_img in enumerate(augmented_images, 1):  # Índice começa em 1
    plt.subplot(3, 3, idx + 1)  # +1 porque o subplot 1 já foi usado
    plt.imshow(np.squeeze(aug_img), cmap='gray')
    plt.title(f'Aug {idx}')
    plt.axis('off')

plt.tight_layout()
plt.show()

# <font face="Verdana" size=3 color='#40E0D0'>  PONTUAÇÃO

* Se o nome do seu colega estiver nos TOP 5 você ganha 1 ponto;
* Se estiver nos TOP 3 você ganha 2 pontos;
* Se estiver nos TOP 2 você ganha 3 pontos;
* Se estiver no  TOP 1 você ganha 5 pontos.